# Curriculum penalty vs penalty cố định
Cùng ngân sách bước, đầu vào xám, train từ đầu (không pretrained), nhiều seed.

| Nhánh | death_penalty / step_penalty |
|---|---|
| `const_low` | 2.5 / 0 suốt run (chỉ train lâu hơn) |
| `const_high` | 20 / 0.1 ngay từ đầu |
| `curriculum` | 2.5/0 -> 10/0.05 -> 20/0.1, tăng dần trong **một** run (`penalty_schedule`) |

Đọc kết quả:
- `const_low` thua cả hai: train lâu hơn không đủ.
- `curriculum` hơn `const_high`: cần học từ từ.
- `curriculum` ngang `const_high`: chỉ cần đặt penalty đúng từ đầu.

In [2]:
from pathlib import Path
from pacman_rl import PPOConfig, piecewise_penalty, train_ppo, get_device
from pacman_rl.runs import save_run

device = get_device()
STEPS = 1_000_000
SEEDS = [0, 1]

ARMS = {
    "const_low":  dict(death_penalty=2.5),
    "const_high": dict(death_penalty=20, step_penalty=0.1),
    "curriculum": dict(penalty_schedule=piecewise_penalty([
        (0.0, 2.5, 0.0), (0.2, 2.5, 0.0),     # 20% đầu: phạt nhẹ để học cách ăn chấm
        (0.5, 10, 0.05),                       # tăng dần
        (0.8, 20, 0.1), (1.0, 20, 0.1),       # 20% cuối: phạt đủ mạnh
    ])),
}

def run_name(arm, seed):
    return f"pen_{arm}_s{seed}"

Mỗi run khoảng 1 giờ. Run đã có file `.pt` thì bỏ qua, nên chạy lại cell này sau khi bị ngắt vẫn tiếp tục được.

In [3]:
for seed in SEEDS:
    for arm, kw in ARMS.items():
        name = run_name(arm, seed)
        if Path(f"runs/{name}.pt").exists() and Path(f"runs/{name}_log.npy").exists():
            print(f"bỏ qua {name} (đã có)")
            continue
        print(f"=== {name} ===")
        cfg = PPOConfig(total_steps=STEPS, lr=2.5e-4, seed=seed, save_every=50_000,
                        n_envs=32, n_steps=512, n_minibatches=4, **kw)
        result = train_ppo(cfg=cfg, model=None, save_path=f"runs/{name}.pt", device=device)
        save_run(result, "runs", name, append=False)

=== pen_const_low_s0 ===
PPO: 61 updates x 16384 mẫu = 999,424 bước, 32 env song song
Update 50/61 | step 819,200 | avg score(20) 790 | 3909 lượt | 210 bước/lượt | 0.28 viên NL/lượt | 0.19 ma/lượt | phạt chết 2.5 bước 0.000 | entropy 1.846 | KL 0.0005 | clipfrac 0.000 | 2824 steps/s
Update 61/61 | step 999,424 | avg score(20) 796 | 868 lượt | 208 bước/lượt | 0.33 viên NL/lượt | 0.22 ma/lượt | phạt chết 2.5 bước 0.000 | entropy 1.732 | KL -0.0000 | clipfrac 0.000 | 2966 steps/s
=== pen_const_high_s0 ===
PPO: 61 updates x 16384 mẫu = 999,424 bước, 32 env song song
Update 50/61 | step 819,200 | avg score(20) 400 | 4339 lượt | 189 bước/lượt | 0.12 viên NL/lượt | 0.08 ma/lượt | phạt chết 20.0 bước 0.100 | entropy 2.067 | KL 0.0004 | clipfrac 0.000 | 3740 steps/s
Update 61/61 | step 999,424 | avg score(20) 515 | 869 lượt | 207 bước/lượt | 0.25 viên NL/lượt | 0.17 ma/lượt | phạt chết 20.0 bước 0.100 | entropy 2.024 | KL 0.0000 | clipfrac 0.000 | 3711 steps/s
=== pen_curriculum_s0 ===
PPO: 61 

In [ ]:
# Setup huấn luyện cho RGB Scale
def run_rgb_name(arm, seed):
    return f"rgb_pen_{arm}_s{seed}"

for seed in SEEDS:
    for arm, kw in ARMS.items():
        name = run_rgb_name(arm, seed)
        if Path(f"runs/{name}.pt").exists() and Path(f"runs/{name}_log.npy").exists():
            print(f"bỏ qua {name} (đã có)")
            continue
        print(f"=== {name} ===")
        cfg = PPOConfig(total_steps=STEPS, lr=2.5e-4, seed=seed, save_every=50_000,
                        n_envs=32, n_steps=512, n_minibatches=4, rgb_scale=True, **kw)
        result = train_ppo(cfg=cfg, model=None, save_path=f"runs/{name}.pt", device=device)
        save_run(result, "runs", name, append=False)

=== rgb_pen_const_low_s0 ===
PPO: 61 updates x 16384 mẫu = 999,424 bước, 32 env song song
Update 50/61 | step 819,200 | avg score(20) 573 | 3884 lượt | 211 bước/lượt | 0.27 viên NL/lượt | 0.19 ma/lượt | phạt chết 2.5 bước 0.000 | entropy 1.720 | KL 0.0009 | clipfrac 0.000 | 2764 steps/s
Update 61/61 | step 999,424 | avg score(20) 662 | 833 lượt | 216 bước/lượt | 0.34 viên NL/lượt | 0.23 ma/lượt | phạt chết 2.5 bước 0.000 | entropy 1.772 | KL 0.0002 | clipfrac 0.000 | 2695 steps/s
=== rgb_pen_const_high_s0 ===
PPO: 61 updates x 16384 mẫu = 999,424 bước, 32 env song song
Update 50/61 | step 819,200 | avg score(20) 406 | 4346 lượt | 188 bước/lượt | 0.11 viên NL/lượt | 0.08 ma/lượt | phạt chết 20.0 bước 0.100 | entropy 2.046 | KL 0.0024 | clipfrac 0.094 | 2373 steps/s
Update 61/61 | step 999,424 | avg score(20) 386 | 854 lượt | 211 bước/lượt | 0.26 viên NL/lượt | 0.20 ma/lượt | phạt chết 20.0 bước 0.100 | entropy 2.007 | KL 0.0000 | clipfrac 0.000 | 2418 steps/s
=== rgb_pen_curriculum_s0 =

c:\Users\LENOVO\miniconda3\envs\rel_pacman\lib\site-packages\gymnasium\vector\async_vector_env.py:460: UserWarning: WARN: Calling `close` while waiting for a pending call to `step` to complete.
  logger.warn(


=== rgb_pen_const_low_s1 ===
PPO: 61 updates x 16384 mẫu = 999,424 bước, 32 env song song


## So sánh

In [ ]:
import numpy as np
from pacman_rl.runs import load_run
from pacman_rl.plotting import compare
from pacman_rl import random_baseline

# names = [run_name(a, s) for a in ARMS for s in SEEDS]
runs = compare(names=None, directory="runs", random_scores=random_baseline(5),
               metrics=("ent",), device=device, save_path="runs/penalty_curriculum.png")

# điểm trung bình 50 ván cuối, gộp theo nhánh
for arm in ARMS:
    m = []
    for s in SEEDS:
        eps = np.asarray(runs[run_name(arm, s)].episodes)
        m.append(eps[-50:, 1].mean())
    print(f"{arm:11s} " + "  ".join(f"{x:7.0f}" for x in m) + f"   TB {np.mean(m):7.0f}")